In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Configure paths and verify prerequisite outputs

In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd

DRIVE = Path('/content/drive/MyDrive')
TRAIN_DIR = DRIVE / 'wesad_task2_outputs' / 'train_processed'
TEST_DIR = DRIVE / 'wesad_task2_outputs' / 'test_processed'
TASK3_DIR = DRIVE / 'wesad_task3_outputs'
TASK4_DIR = DRIVE / 'wesad_task4_outputs'
TASK5_DIR = DRIVE / 'wesad_task5_outputs'
EXPORT_DIR = DRIVE / 'wesad_task6_tableau_exports'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

required_paths = [
    TRAIN_DIR, TEST_DIR,
    TASK3_DIR / 'task3_model_comparison.csv',
    TASK4_DIR / 'task4_benchmark.json',
    TASK5_DIR / 'task5_model_metrics.csv',
    TASK5_DIR / 'task5_perturbation_stability.csv',
]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Required outputs are missing. Run the final Task 2, Task 3, Task 4 and Task 5 notebooks successfully first. Missing:\n- '
        + '\n- '.join(missing)
    )
print('All required Task 2–5 output paths are present.')
print('Tableau export directory:', EXPORT_DIR)


All required Task 2–5 output paths are present.
Tableau export directory: /content/drive/MyDrive/wesad_task6_tableau_exports


## Data quality & pipeline monitoring

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName('WESAD_Task6_Tableau_Final').getOrCreate()
train_df = spark.read.parquet(str(TRAIN_DIR))
test_df = spark.read.parquet(str(TEST_DIR))

def parquet_profile(label, folder):
    folder = Path(folder)
    files = list(folder.rglob('*.parquet'))
    return {
        'Pipeline_Stage': label,
        'Folder_Available': folder.exists(),
        'Parquet_File_Count': len(files),
        'Stored_Size_MB': round(sum(p.stat().st_size for p in files) / (1024**2), 3)
    }

pipeline_df = pd.DataFrame([
    parquet_profile('Processed training split', TRAIN_DIR),
    parquet_profile('Processed held-out test split', TEST_DIR)
])
pipeline_df.to_csv(EXPORT_DIR / 'dashboard1_pipeline_storage.csv', index=False)

def export_class_summary(sdf, filename, source_label):
    result = sdf.groupBy('target_text').count().withColumnRenamed('count', 'Record_Count').toPandas()
    total = result['Record_Count'].sum()
    result['Share_Percent'] = (100 * result['Record_Count'] / total).round(2) if total else 0
    result['Data_Source'] = source_label
    result.to_csv(EXPORT_DIR / filename, index=False)

export_class_summary(train_df, 'dashboard1_training_class_distribution.csv', 'Processed training split')
export_class_summary(test_df, 'dashboard1_test_class_distribution.csv', 'Processed held-out test split')

null_rows = []
for split_name, sdf in [('Training', train_df), ('Held-out test', test_df)]:
    values = sdf.agg(*[
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in sdf.columns
    ]).first().asDict()
    for column, count in values.items():
        null_rows.append({'Split': split_name, 'Column': column, 'Null_Count': int(count or 0)})
pd.DataFrame(null_rows).to_csv(EXPORT_DIR / 'dashboard1_null_profile.csv', index=False)

print('Dashboard 1 exports created.')
display(pipeline_df)


Dashboard 1 exports created.


,Pipeline_Stage,Folder_Available,Parquet_File_Count,Stored_Size_MB
0,Processed training split,True,16,505.253
1,Processed held-out test split,True,14,126.471


#Business insights

In [ ]:
participant_df = (
    test_df.groupBy('subject_id', 'target_text')
    .count()
    .withColumnRenamed('count', 'Record_Count')
    .toPandas()
)
participant_df['Participant_Total'] = participant_df.groupby('subject_id')['Record_Count'].transform('sum')
participant_df['Class_Share_Percent'] = (
    100 * participant_df['Record_Count'] / participant_df['Participant_Total']
).round(2)
participant_df.to_csv(EXPORT_DIR / 'dashboard3_participant_stress_profile.csv', index=False)

class_business = (
    test_df.groupBy('target_text').count()
    .withColumnRenamed('count', 'Record_Count')
    .toPandas()
)
class_business['Share_Percent'] = (
    100 * class_business['Record_Count'] / class_business['Record_Count'].sum()
).round(2)
class_business.to_csv(EXPORT_DIR / 'dashboard3_heldout_class_summary.csv', index=False)

print('Dashboard 3 exports created.')
display(participant_df.head(10))


Dashboard 3 exports created.


,subject_id,target_text,Record_Count,Participant_Total,Class_Share_Percent
0,S16,stress,471101,2109100,22.34
1,S16,non_stress,1637999,2109100,77.66
2,S17,non_stress,1598800,2104900,75.96
3,S17,stress,506100,2104900,24.04
4,S15,non_stress,1638699,2118899,77.34
5,S15,stress,480200,2118899,22.66


## Model performance & feature importance

In [ ]:
task3_metrics = pd.read_csv(TASK3_DIR / 'task3_model_comparison.csv')
task5_metrics = pd.read_csv(TASK5_DIR / 'task5_model_metrics.csv')

def normalise_columns(df):
    out = df.copy()
    out.columns = [
        re.sub(r'[^a-z0-9]+', '_', str(c).strip().lower()).strip('_')
        for c in out.columns
    ]
    return out

import re
task3_metrics = normalise_columns(task3_metrics)
task5_metrics = normalise_columns(task5_metrics)

if 'model' not in task3_metrics.columns or len(task3_metrics) == 0:
    raise ValueError('Task 3 model-comparison CSV is empty or lacks a Model column.')
if 'model' not in task5_metrics.columns or len(task5_metrics) == 0:
    raise ValueError('Task 5 model-metrics CSV is empty or lacks a Model column.')

task3_metrics.insert(0, 'Source_Task', 'Task 3')
task5_metrics.insert(0, 'Source_Task', 'Task 5')
task3_metrics.to_csv(EXPORT_DIR / 'dashboard2_task3_model_performance.csv', index=False)
task5_metrics.to_csv(EXPORT_DIR / 'dashboard2_task5_model_performance.csv', index=False)

stability = pd.read_csv(TASK5_DIR / 'task5_perturbation_stability.csv')
stability.to_csv(EXPORT_DIR / 'dashboard2_perturbation_stability.csv', index=False)

print('Task 3 model metrics:', task3_metrics.shape)
print('Task 5 model metrics:', task5_metrics.shape)
print('Task 5 perturbation rows:', len(stability))
display(task5_metrics)


Task 3 model metrics: (4, 12)
Task 5 model metrics: (4, 16)
Task 5 perturbation rows: 12


,Source_Task,model,accuracy,roc_auc,pr_auc,weighted_precision,weighted_recall,weighted_f1,stress_precision,stress_recall,decision_threshold,tn,fp,fn,tp,training_time_seconds
0,Task 5,Gradient-Boosted Trees (GBT) Classifier,0.85219,0.926488,0.774326,0.846123,0.85219,0.847844,0.709860,0.603993,0.50,71332,5676,9105,13887,151.55
1,Task 5,Random Forest Classifier,0.87044,0.923737,0.787527,0.865954,0.87044,0.861282,0.818078,0.561326,0.50,74138,2870,10086,12906,70.36
2,Task 5,Logistic Regression,0.48995,0.622352,0.382297,0.670211,0.48995,0.526930,0.250004,0.609212,0.24,34988,42020,8985,14007,77.46
3,Task 5,Multilayer Perceptron Classifier,0.47645,0.536612,0.225583,0.646224,0.47645,0.515616,0.230214,0.544885,0.24,35117,41891,10464,12528,115.50


## Export genuine feature-importance evidence

In [ ]:
# Export real feature-importance evidence from the saved Task 3 best model.
# Feature names are indexed because Task 2 stores predictors as a Spark feature vector
# and does not preserve a verified human-readable name for every vector position.
from pyspark.ml.classification import (
    LogisticRegressionModel, RandomForestClassificationModel,
    GBTClassificationModel, MultilayerPerceptronClassificationModel
)

task3_meta_path = TASK3_DIR / 'task3_metadata.json'
if task3_meta_path.exists():
    task3_meta = json.loads(task3_meta_path.read_text(encoding='utf-8'))
    best_model_name = str(task3_meta.get('best_model_by_roc_auc', ''))
else:
    best_model_name = str(task3_metrics.sort_values(
        'roc_auc' if 'roc_auc' in task3_metrics.columns else task3_metrics.columns[1],
        ascending=False
    ).iloc[0]['model'])

best_model_path = TASK3_DIR / 'best_model'
if not best_model_path.exists():
    raise FileNotFoundError(f'Saved Task 3 best model not found: {best_model_path}')

if 'Random Forest' in best_model_name:
    explain_model = RandomForestClassificationModel.load(str(best_model_path))
    importance_values = np.asarray(explain_model.featureImportances.toArray(), dtype=float)
    importance_method = 'Saved Task 3 Random Forest featureImportances'
elif 'Gradient-Boosted' in best_model_name or 'GBT' in best_model_name:
    explain_model = GBTClassificationModel.load(str(best_model_path))
    importance_values = np.asarray(explain_model.featureImportances.toArray(), dtype=float)
    importance_method = 'Saved Task 3 GBT featureImportances'
elif 'Logistic Regression' in best_model_name:
    explain_model = LogisticRegressionModel.load(str(best_model_path))
    importance_values = np.abs(np.asarray(explain_model.coefficients.toArray(), dtype=float))
    importance_method = 'Absolute coefficients from saved Task 3 Logistic Regression'
else:
    explain_model = MultilayerPerceptronClassificationModel.load(str(best_model_path))
    # For the MLP, compute a transparent surrogate explanation using predictions from
    # the saved best model. These importances explain the surrogate approximation.
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score
    from pyspark.ml.linalg import Vectors
    sample_rows = train_df.select('features').limit(8000).collect()
    X_surrogate = np.vstack([np.asarray(r['features'].toArray(), dtype=float) for r in sample_rows])
    pred_input = spark.createDataFrame(
        [(Vectors.dense(x.tolist()),) for x in X_surrogate], ['features']
    )
    y_surrogate = np.asarray([
        int(r['prediction']) for r in explain_model.transform(pred_input).select('prediction').collect()
    ])
    surrogate_model = RandomForestClassifier(
        n_estimators=100, max_depth=8, random_state=42, n_jobs=-1
    )
    surrogate_model.fit(X_surrogate, y_surrogate)
    importance_values = np.asarray(surrogate_model.feature_importances_, dtype=float)
    importance_method = 'Random Forest surrogate feature importance for saved Task 3 MLP'

feature_importance = pd.DataFrame({
    'Feature': [f'Feature_{i+1}' for i in range(len(importance_values))],
    'Importance': importance_values,
    'Importance_Method': importance_method
}).sort_values('Importance', ascending=False)
feature_importance.to_csv(EXPORT_DIR / 'dashboard2_feature_importance.csv', index=False)
print('Saved genuine model explanation:', importance_method)
display(feature_importance.head(15))


Saved genuine model explanation: Saved Task 3 GBT featureImportances


,Feature,Importance,Importance_Method
5,Feature_6,0.331015,Saved Task 3 GBT featureImportances
6,Feature_7,0.314434,Saved Task 3 GBT featureImportances
2,Feature_3,0.173959,Saved Task 3 GBT featureImportances
0,Feature_1,0.096762,Saved Task 3 GBT featureImportances
1,Feature_2,0.081240,Saved Task 3 GBT featureImportances
7,Feature_8,0.001750,Saved Task 3 GBT featureImportances
3,Feature_4,0.000816,Saved Task 3 GBT featureImportances
4,Feature_5,0.000024,Saved Task 3 GBT featureImportances


## Scalability & cost analysis

In [ ]:
with open(TASK4_DIR / 'task4_benchmark.json', 'r', encoding='utf-8') as f:
    benchmark = json.load(f)

benchmark_df = pd.json_normalize(benchmark)
benchmark_df.to_csv(EXPORT_DIR / 'dashboard4_task4_scalability_benchmark.csv', index=False)

scalability_df = pipeline_df.copy()
scalability_df['Operational_Metric'] = 'Stored Parquet footprint'
scalability_df['Monetary_Cost'] = np.nan
scalability_df['Cost_Data_Status'] = 'No cloud billing data recorded; monetary costs are not inferred'
scalability_df.to_csv(EXPORT_DIR / 'dashboard4_scalability_storage.csv', index=False)

print('Task 4 benchmark evidence exported.')
display(benchmark_df)


Task 4 benchmark evidence exported.


,spark_version,application_id,spark_ui_url,spark_master,driver_memory,executor_memory,executor_cores,default_parallelism,shuffle_partitions,adaptive_execution,...,repartitioned_partitions,target_repartition_count,baseline_aggregation_seconds,repartitioned_aggregation_seconds,cache_materialisation_seconds,cached_repeat_aggregation_seconds,uncached_repeat_aggregation_seconds,benchmark_scope,spark_ui_screenshot_required,status
0,4.0.4,local-1793018980689,http://53b1f0562d21:4040,local[*],Spark default; not explicitly set,Spark default; not explicitly set,Local mode; executor cores not separately conf...,2,8,true,...,8,8,1.1851,0.7479,0.5273,0.703,5.3967,bounded deterministic sample; not a full-datas...,True,completed


## Final export validation

In [ ]:
required_exports = [
    'dashboard1_pipeline_storage.csv',
    'dashboard1_training_class_distribution.csv',
    'dashboard1_test_class_distribution.csv',
    'dashboard1_null_profile.csv',
    'dashboard2_task3_model_performance.csv',
    'dashboard2_task5_model_performance.csv',
    'dashboard2_perturbation_stability.csv',
    'dashboard3_participant_stress_profile.csv',
    'dashboard3_heldout_class_summary.csv',
    'dashboard4_task4_scalability_benchmark.csv',
    'dashboard4_scalability_storage.csv',
]
missing_or_empty = [
    name for name in required_exports
    if not (EXPORT_DIR / name).exists() or (EXPORT_DIR / name).stat().st_size == 0
]
if missing_or_empty:
    raise RuntimeError('Export validation failed for:\n- ' + '\n- '.join(missing_or_empty))

inventory = []
for path in sorted(EXPORT_DIR.glob('*.csv')):
    try:
        frame = pd.read_csv(path)
        inventory.append({
            'File_Name': path.name,
            'Rows': len(frame),
            'Columns': len(frame.columns),
            'Size_KB': round(path.stat().st_size / 1024, 2),
            'Non_Empty': path.stat().st_size > 0
        })
    except (pd.errors.EmptyDataError, pd.errors.ParserError):
        inventory.append({
            'File_Name': path.name, 'Rows': 0, 'Columns': 0,
            'Size_KB': round(path.stat().st_size / 1024, 2), 'Non_Empty': False
        })

inventory_df = pd.DataFrame(inventory)
inventory_df.to_csv(EXPORT_DIR / 'tableau_export_inventory.csv', index=False)
print(f'PASS: {len(required_exports)} required exports exist and are non-empty.')
print('Export folder:', EXPORT_DIR)
display(inventory_df)


PASS: 11 required exports exist and are non-empty.
Export folder: /content/drive/MyDrive/wesad_task6_tableau_exports


,File_Name,Rows,Columns,Size_KB,Non_Empty
0,README_tableau_exports.csv,7,3,0.97,True
1,dashboard1_class_distribution.csv,2,4,0.16,True
2,dashboard1_null_profile.csv,10,3,0.25,True
3,dashboard1_pipeline_storage.csv,2,4,0.15,True
4,dashboard1_sample_class_distribution.csv,2,4,0.16,True
5,dashboard1_test_class_distribution.csv,2,4,0.15,True
6,dashboard1_training_class_distribution.csv,2,4,0.14,True
7,dashboard2_feature_importance.csv,8,3,0.56,True
8,dashboard2_perturbation_stability.csv,12,4,0.71,True
9,dashboard2_task3_model_performance.csv,4,12,0.70,True
